# How to use `data.splits`

Basins can overlap in two ways: spatially (in which case one is contained in the other) and temporally (if basin A is contained in basin B but A ran from 2008 - 2010 and basin B ran from 2020-2024, they don't actually overlap). Use `data.splits` to generate folds and train/test splits to avoid data leakage.

Below is the TLDR copy and paste version, after that is a little more info.

In [ ]:
import sys
sys.path.insert(0, "../")
from src.splits.conflict_graph import make_folds, audit_split, holdout_split

In [ ]:
# -------------------------------------
# Your main k-fold loop: use make_folds
# -------------------------------------
num_folds = 4
random_state = 42

# the thing that does the thing
folds = make_folds(n_splits=num_folds, shuffle=True, random_state=random_state)

for i in range(num_folds):
    test_sites = folds.index[folds.fold == i].tolist()
    train_sites = folds.index[folds.fold != i].tolist()
    
    print(f" Have {len(train_sites)} train sites and {len(test_sites)} test sites")
    print(f"  first 5 train sites: {train_sites[:5]}")
    print(f"  first 5 test sites: {test_sites[:5]}")
    
    # makes sure the sites don't exhibit data leakage
    assert audit_split(train_sites, test_sites).query("severity == 'hard'").empty
   
print("--------------") 
# -------------------------
# If you want a quick split
# -------------------------

train_sites, test_sites = holdout_split(test_size=0.2, seed=0)
print(f"Have {len(train_sites)} train sites and {len(test_sites)} test sites")
print(f"Does the audit pass? {audit_split(train_sites, test_sites).query("severity == 'hard'").empty}")

Here's an exploration of what's happening here if you care.
## List of all methods in `splits.py` low level -> high level
They compose in a chain, I don't know why you'd ever call more than the last three.

* `nitrate_span(site)` -> a site's `(first, last)` nitrate date (exists on temporal axis)
* `overlaps(span_a, span_b, buffer)` -> do two records overlap in time, within the autocorrelation buffer (tests for temporal leak)
* `build_conflict_graph(buffer)` -> a graph where an edge means "these two sites are spatially related and temporally overlapping" (a hard leak) (spatial axis from `get_basin_graph`, intersected with the temporal test)
* `split_groups(buffer)` -> `{site: group_id}`. Each connected component of the conflict graph is an indivisible group that must stay on the same side of the train/test split.
* `make_folds(...)` / `holdout_split(...)` -> the two things you actually call.
* audit_split(...) -> the correctness/QA check.

## The fold method: `make_folds`

This produces a dataframe indexed by `site_uid`s with two columns: `group` and `fold`.
* `group` is a number identifying the connected component of the conflict graph on which this site lives. The conflict graph has sites as its nodes, and an edge between A and B if A and B both overlap spatially AND temporally.
* `fold` is the number giving the fold of the site.

In [ ]:
folds = make_folds(n_splits=5) # the thing that does the thing
print(folds.head(10))

In [ ]:
test_sites = folds.index[folds.fold == 2] # get the test group 2
train_sites = folds.index[folds.fold != 2] # everything else is the training group
# calling folds.index means you'll get nothing but the site_uid

# this returns a dataframe with one row per pair which overlap in either time or space
audit = audit_split(train_sites, test_sites)
print(audit.head())

# severity is either spatial, temporal, or hard. The first two or fine, the last isn't, so we do a check:
assert audit[audit.severity == "hard"].empty

# this is a one-liner to wrap this whole audit into one line
assert audit_split(train_sites, test_sites).query("severity == 'hard'").empty